In [ ]:
!pip install tensorflow matplotlib numpy pandas scikit-learn

In [ ]:
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Path to scraped images 
data_dir = "/content/drive/MyDrive/ds_task/CNN_IMAGES"
# Path to save the model after training
model_path = "/content/drive/MyDrive/ds_task/cnn_product_model.h5"

# Image parameters
img_height, img_width = 128, 128
batch_size = 16

# Data augmentation & preprocessing
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,  # 80% train, 20% val
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

# Data loading
train_generator = train_datagen.flow_from_directory(
    data_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical',
    subset='training'
)

val_generator = train_datagen.flow_from_directory(
    data_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation'
)

# Number of classes
num_classes = len(train_generator.class_indices)
print("Classes:", train_generator.class_indices)

# Build CNN model from scratch with 384-dim embeddings
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(img_height, img_width,3)),
    MaxPooling2D((2,2)),
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D((2,2)),
    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D((2,2)),
    Flatten(),
    Dense(384, activation='relu'),   # <- changed from 128 to 384 dimension
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

# Callbacks and save
callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True),
    ModelCheckpoint(model_path, save_best_only=True)
]

# Train the model
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=30,
    callbacks=callbacks
)

# confirmation of success save 
print(f"Model saved to {model_path}")

Found 230 images belonging to 10 classes.
Found 54 images belonging to 10 classes.
Classes: {'20726': 0, '21034': 1, '21931': 2, '22077': 3, '22112': 4, '22139': 5, '22384': 6, '22423': 7, '22727': 8, '23298': 9}
Model: "sequential_3"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_9 (Conv2D)               │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_10 (Conv2D)              │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_11 (Conv2D)              │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_11 (MaxPooling2D) │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 384)            │     9,634,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 384)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 10)             │         3,850 │
└─────────────────────────────────┴────────────────────────┴───────────────┘
 Total params: 9,731,274 (37.12 MB)
 Trainable params: 9,731,274 (37.12 MB)
 Non-trainable params: 0 (0.00 B)
Epoch 1/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 737ms/step - accuracy: 0.1320 - loss: 2.4330
WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. 
15/15 ━━━━━━━━━━━━━━━━━━━━ 16s 926ms/step - accuracy: 0.1317 - loss: 2.4295 - val_accuracy: 0.2037 - val_loss: 2.2230
Epoch 2/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 740ms/step - accuracy: 0.1725 - loss: 2.2446
WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. 
15/15 ━━━━━━━━━━━━━━━━━━━━ 15s 983ms/step - accuracy: 0.1734 - loss: 2.2428 - val_accuracy: 0.2407 - val_loss: 2.0157
Epoch 3/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 767ms/step - accuracy: 0.2232 - loss: 2.1536
WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. 
15/15 ━━━━━━━━━━━━━━━━━━━━ 16s 1s/step - accuracy: 0.2231 - loss: 2.1548 - val_accuracy: 0.3333 - val_loss: 1.9037
Epoch 4/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.2790 - loss: 1.9863
WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. 
15/15 ━━━━━━━━━━━━━━━━━━━━ 22s 1s/step - accuracy: 0.2787 - loss: 1.9896 - val_accuracy: 0.3704 - val_loss: 1.8088
Epoch 5/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 960ms/step - accuracy: 0.2542 - loss: 1.9982
WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. 
15/15 ━━━━━━━━━━━━━━━━━━━━ 18s 1s/step - accuracy: 0.2562 - loss: 1.9962 - val_accuracy: 0.3704 - val_loss: 1.7762
Epoch 6/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 719ms/step - accuracy: 0.3872 - loss: 1.8656
WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. 
15/15 ━━━━━━━━━━━━━━━━━━━━ 16s 1s/step - accuracy: 0.3836 - loss: 1.8700 - val_accuracy: 0.3704 - val_loss: 1.7189
Epoch 7/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 17s 1s/step - accuracy: 0.2743 - loss: 1.9876 - val_accuracy: 0.4259 - val_loss: 1.7591
Epoch 8/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 847ms/step - accuracy: 0.3325 - loss: 1.8330
WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. 
15/15 ━━━━━━━━━━━━━━━━━━━━ 15s 1s/step - accuracy: 0.3326 - loss: 1.8333 - val_accuracy: 0.4444 - val_loss: 1.6758
Epoch 9/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 977ms/step - accuracy: 0.4147 - loss: 1.6657
WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. 
15/15 ━━━━━━━━━━━━━━━━━━━━ 16s 1s/step - accuracy: 0.4127 - loss: 1.6691 - val_accuracy: 0.4444 - val_loss: 1.6706
Epoch 10/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.3378 - loss: 1.7278
WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. 
15/15 ━━━━━━━━━━━━━━━━━━━━ 25s 1s/step - accuracy: 0.3360 - loss: 1.7296 - val_accuracy: 0.4074 - val_loss: 1.6511
Epoch 11/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 21s 1s/step - accuracy: 0.4272 - loss: 1.6814 - val_accuracy: 0.2778 - val_loss: 1.8547
Epoch 12/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 13s 843ms/step - accuracy: 0.3453 - loss: 1.7445 - val_accuracy: 0.3519 - val_loss: 1.7879
Epoch 13/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 789ms/step - accuracy: 0.4453 - loss: 1.4823
WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. 
15/15 ━━━━━━━━━━━━━━━━━━━━ 15s 1s/step - accuracy: 0.4427 - loss: 1.4871 - val_accuracy: 0.3704 - val_loss: 1.6240
Epoch 14/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 16s 1s/step - accuracy: 0.5023 - loss: 1.4928 - val_accuracy: 0.2963 - val_loss: 1.6811
Epoch 15/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 13s 878ms/step - accuracy: 0.5318 - loss: 1.4182 - val_accuracy: 0.4259 - val_loss: 1.7638
Epoch 16/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 760ms/step - accuracy: 0.4447 - loss: 1.5131
WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. 
15/15 ━━━━━━━━━━━━━━━━━━━━ 15s 976ms/step - accuracy: 0.4451 - loss: 1.5142 - val_accuracy: 0.5185 - val_loss: 1.5122
Epoch 17/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 16s 1s/step - accuracy: 0.5316 - loss: 1.4042 - val_accuracy: 0.3333 - val_loss: 1.7511
Epoch 18/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 13s 862ms/step - accuracy: 0.4383 - loss: 1.3954 - val_accuracy: 0.4444 - val_loss: 1.6500
Epoch 19/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 11s 745ms/step - accuracy: 0.5306 - loss: 1.2210 - val_accuracy: 0.5185 - val_loss: 1.6770
Epoch 20/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 782ms/step - accuracy: 0.5988 - loss: 1.2012
WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. 
15/15 ━━━━━━━━━━━━━━━━━━━━ 24s 978ms/step - accuracy: 0.5970 - loss: 1.2102 - val_accuracy: 0.4630 - val_loss: 1.4940
Epoch 21/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 16s 1s/step - accuracy: 0.5357 - loss: 1.2167 - val_accuracy: 0.4815 - val_loss: 1.6627
Epoch 22/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 16s 723ms/step - accuracy: 0.6332 - loss: 1.1263 - val_accuracy: 0.4630 - val_loss: 1.6723
Epoch 23/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 12s 802ms/step - accuracy: 0.5748 - loss: 1.1182 - val_accuracy: 0.5000 - val_loss: 1.7442
Epoch 24/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 789ms/step - accuracy: 0.5917 - loss: 1.1361
WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. 
15/15 ━━━━━━━━━━━━━━━━━━━━ 15s 979ms/step - accuracy: 0.5909 - loss: 1.1368 - val_accuracy: 0.5185 - val_loss: 1.4655
Epoch 25/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 22s 1s/step - accuracy: 0.5919 - loss: 1.0529 - val_accuracy: 0.4074 - val_loss: 1.7982
Epoch 26/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 12s 792ms/step - accuracy: 0.6088 - loss: 1.1116 - val_accuracy: 0.5370 - val_loss: 1.7992
Epoch 27/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 11s 718ms/step - accuracy: 0.6465 - loss: 0.9995 - val_accuracy: 0.3519 - val_loss: 2.2231
Epoch 28/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 13s 842ms/step - accuracy: 0.6752 - loss: 0.9593 - val_accuracy: 0.5185 - val_loss: 1.8315
Epoch 29/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 13s 893ms/step - accuracy: 0.6135 - loss: 1.0529 - val_accuracy: 0.5370 - val_loss: 1.6671
Model saved to /content/drive/MyDrive/ds_task/cnn_product_model.h5